## Production-Style Sequence Packing

Large-scale GPT pretraining generally avoids padding every document to the maximum sequence length. Instead, documents are tokenized with an explicit end-of-document boundary such as EOS/EOD, concatenated into a token stream, and then packed into fixed-length training sequences.

For our TinyStories dataset:

```text
Story A + EOS
Story B + EOS
Story C + EOS
       ↓
Concatenate token IDs
       ↓
Split into chunks of MAX_SEQ_LEN
       ↓
Create input/target pairs
       ↓
[B, MAX_SEQ_LEN]

In [1]:
import json
import torch

import sys
sys.path.append("..")

from config import TOKENIZER_PATH, MAX_SEQ_LEN
from tokenizer.bpe import ByteLevelBPETokenizer

tokenizer = ByteLevelBPETokenizer()
tokenizer.load("../" + TOKENIZER_PATH)

all_token_ids = []

with open("../data/raw/train.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        story = json.loads(line)["text"]

        token_ids = tokenizer.encode(story)

        # Mark the end of each story.
        token_ids.append(tokenizer.special_tokens["<|endoftext|>"])

        all_token_ids.extend(token_ids)

print("Total tokens:", len(all_token_ids))

Total tokens: 4932617


In [7]:
# Convert to tensor and create chunks

tokens = torch.tensor(all_token_ids, dtype=torch.long)
# print(tokens[:1000])

num_full_chunks = len(tokens) // MAX_SEQ_LEN

tokens = tokens[:num_full_chunks * MAX_SEQ_LEN]

chunks = tokens.reshape(num_full_chunks, MAX_SEQ_LEN)

print("Token stream shape:", tokens.shape)
print("Number of chunks:", chunks.shape[0])
print("Chunk shape:", chunks.shape)

Token stream shape: torch.Size([4932608])
Number of chunks: 9634
Chunk shape: torch.Size([9634, 512])


In [8]:
eos_id = tokenizer.special_tokens["<|endoftext|>"]

eos_count = (chunks == eos_id).sum().item()

print("EOS tokens in chunks:", eos_count)
print("EOS tokens in original stream:", all_token_ids.count(eos_id))

EOS tokens in chunks: 24999
EOS tokens in original stream: 25000


In [19]:
input_tokens = tokens[:-1]
target_tokens = tokens[1:]

num_sequences = len(input_tokens) // MAX_SEQ_LEN
print(num_sequences)

input_tokens = input_tokens[:num_sequences * MAX_SEQ_LEN]
target_tokens = target_tokens[:num_sequences * MAX_SEQ_LEN]

print('input_tokens: ', input_tokens.shape)
print('target_tokens: ', target_tokens.shape)

input_chunks = input_tokens.reshape(num_sequences, MAX_SEQ_LEN)
target_chunks = target_tokens.reshape(num_sequences, MAX_SEQ_LEN)

print("Input shape: ", input_chunks.shape)
print("Target shape:", target_chunks.shape)

9633
input_tokens:  torch.Size([4932096])
target_tokens:  torch.Size([4932096])
Input shape:  torch.Size([9633, 512])
Target shape: torch.Size([9633, 512])


In [26]:
input_chunks[0, 1:].item

<function Tensor.item>

In [29]:
print("Input shape: ", input_chunks.shape)
print("Target shape:", target_chunks.shape)

# Verify the target is shifted by one token
assert torch.equal(input_chunks[0, 1:], target_chunks[0, :-1])

print("Input/target alignment verified.")
print("Chunk-boundary prediction verified.")

Input shape:  torch.Size([9633, 512])
Target shape: torch.Size([9633, 512])
Input/target alignment verified.
Chunk-boundary prediction verified.


In [30]:
print("Input: ")
print(input_chunks[0])

print("\nTarget:")
print(target_chunks[0])

Input: 
tensor([ 430,  437,  259,  398,   44,  314,  259,  558,  593,   44,  400,  630,
         259, 2781,  464,  401,  587,   46,  587,  524,  266,  323,  267,  739,
         431,  328,   46,  952,  262, 4359,   44,  636,  262,  632,  428,  606,
          44,  587,  528,  651,  377, 1149,   46,   10,  382, 4359,   44,  877,
         259,  741,  328,  369,  579,   44,  587,  283,  377, 2999,   46,  315,
         407,  266, 1817,  602,  678,   46,  695,   44,  286,  428,  266,  340,
        1611,  266, 1817,   46,  537,   44,  712,  286,  283,  390, 2781,   44,
         286, 2611,  262, 1611,  883,  267,  678,  428, 1636,   33,   10, 1612,
         531,  887,  665,  994, 2781,   44,  411,  340, 1916,   44,  259,  389,
         473,  401, 1692,   44,  804,  702,   46,  341,  322,   44,   32,   34,
        5807,  587,   44, 3010,  697,  390, 2781,  406,  341, 2349,  368,  262,
         678,  267, 1788,  262, 1611,  591,   46, 1264,  398,   44,  587,  283,
         377,  838,  267, 2714, 